In [31]:
!pip install -q segmentation-models-pytorch==0.3.4 albumentations==1.4.3

In [32]:
import kagglehub
briscdataset_brisc2025_path = kagglehub.dataset_download('briscdataset/brisc2025')
print('Data source import complete.')

Data source import complete.


In [33]:
import os, glob, random, warnings, hashlib
import numpy as np
import pandas as pd
import cv2
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import models

import segmentation_models_pytorch as smp
import albumentations as A
from albumentations.pytorch import ToTensorV2

warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)

Device: cuda


In [34]:
DATA_ROOT = "/kaggle/input/datasets/briscdataset/brisc2025/brisc2025"

CLS_TRAIN_DIR  = os.path.join(DATA_ROOT, "classification_task", "train")
CLS_TEST_DIR   = os.path.join(DATA_ROOT, "classification_task", "test")

SEG_TRAIN_IMG  = os.path.join(DATA_ROOT, "segmentation_task", "train", "images")
SEG_TRAIN_MASK = os.path.join(DATA_ROOT, "segmentation_task", "train", "masks")
SEG_TEST_IMG   = os.path.join(DATA_ROOT, "segmentation_task", "test",  "images")
SEG_TEST_MASK  = os.path.join(DATA_ROOT, "segmentation_task", "test",  "masks")

CLASS_NAMES  = ["glioma", "meningioma", "no_tumor", "pituitary"]
CLASS_TO_IDX = {c: i for i, c in enumerate(CLASS_NAMES)}
NUM_CLASSES  = len(CLASS_NAMES)
IMG_SIZE     = 224

In [35]:
def build_cls_df(root_dir):
    rows = []
    for cls_name in CLASS_NAMES:
        cls_dir = os.path.join(root_dir, cls_name)
        if not os.path.isdir(cls_dir):
            continue
        for ext in ("*.jpg", "*.png", "*.jpeg"):
            for fpath in glob.glob(os.path.join(cls_dir, ext)):
                rows.append({"filepath": fpath, "label_name": cls_name, "label": CLASS_TO_IDX[cls_name]})
    return pd.DataFrame(rows)

cls_full_df = build_cls_df(CLS_TRAIN_DIR)
cls_test_df = build_cls_df(CLS_TEST_DIR)
print("Initial train:", len(cls_full_df), "| Initial test:", len(cls_test_df))

def file_hash(path):
    with open(path, 'rb') as f:
        return hashlib.md5(f.read()).hexdigest()

print("Checking duplicates...")
train_hashes = {file_hash(p): p for p in cls_full_df["filepath"]}
duplicates   = [p for p in cls_test_df["filepath"] if file_hash(p) in train_hashes]
print("Duplicates found:", len(duplicates))

cls_test_df = cls_test_df[~cls_test_df["filepath"].isin(duplicates)].reset_index(drop=True)
print("Test after cleaning:", len(cls_test_df))

cls_train_df, cls_val_df = train_test_split(
    cls_full_df, test_size=0.2, stratify=cls_full_df["label"], random_state=SEED
)
print("Train:", len(cls_train_df), "| Val:", len(cls_val_df))

Initial train: 5000 | Initial test: 1000
Checking duplicates...
Duplicates found: 7
Test after cleaning: 993
Train: 4000 | Val: 1000


In [36]:
def build_seg_df(img_dir, mask_dir):
    rows = []
    for img_path in sorted(glob.glob(os.path.join(img_dir, "*.jpg"))):
        stem      = os.path.splitext(os.path.basename(img_path))[0]
        mask_path = os.path.join(mask_dir, stem + ".png")
        if os.path.exists(mask_path):
            rows.append({"img_path": img_path, "mask_path": mask_path})
    return pd.DataFrame(rows)

seg_train_df = build_seg_df(SEG_TRAIN_IMG, SEG_TRAIN_MASK)
seg_test_df  = build_seg_df(SEG_TEST_IMG,  SEG_TEST_MASK)
print("Seg train:", len(seg_train_df), "| Seg test:", len(seg_test_df))

Seg train: 3933 | Seg test: 860


In [37]:
# ============================================================================
# DATA PIPELINES
# ============================================================================
IMG_SIZE = 224

# We create 4 distinct training pipelines.
# Validation/Test will ALWAYS use the clean "Raw" pipeline (no augmentations, no CLAHE).

# 1. Baseline (Raw)
pipeline_raw = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE),
    A.ToRGB(),  # Ensures 3 channels for ImageNet weights
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2(),
])

# 2. Augmented (Geometric transforms)
pipeline_aug = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE),
    A.ToRGB(),
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
    A.RandomRotate90(p=0.5),
    A.ShiftScaleRotate(shift_limit=0.05, scale_limit=0.05, rotate_limit=15, p=0.5),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2(),
])

# 3. Raw + CLAHE (Contrast enhancement)
# Albumentations automatically converts RGB -> LAB, applies CLAHE to L-channel, then LAB -> RGB
pipeline_clahe_raw = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE),
    A.ToRGB(),
    A.CLAHE(clip_limit=2.0, tile_grid_size=(8, 8), p=1.0),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2(),
])

# 4. Augmented + CLAHE
pipeline_clahe_aug = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE),
    A.ToRGB(),
    A.CLAHE(clip_limit=2.0, tile_grid_size=(8, 8), p=1.0),
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
    A.RandomRotate90(p=0.5),
    A.ShiftScaleRotate(shift_limit=0.05, scale_limit=0.05, rotate_limit=15, p=0.5),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2(),
])

# Dictionary to loop over
PIPELINES = {
    "Raw": pipeline_raw,
    "Augmented": pipeline_aug,
    "Raw_CLAHE": pipeline_clahe_raw,
    "Augmented_CLAHE": pipeline_clahe_aug
}

# Validation pipeline is ALWAYS the clean raw pipeline
test_aug = pipeline_raw

print(f"Defined {len(PIPELINES)} training pipelines: {list(PIPELINES.keys())}")

Defined 4 training pipelines: ['Raw', 'Augmented', 'Raw_CLAHE', 'Augmented_CLAHE']


In [38]:
# ============================================================================
# DATASET CLASSES (Unchanged logic)
# ============================================================================
class BrainClassDataset(Dataset):
    def __init__(self, df, transform=None):
        self.df        = df.reset_index(drop=True)
        self.transform = transform

    def __len__(self): return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = cv2.imread(row["filepath"])
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        if self.transform:
            img = self.transform(image=img)["image"]
        return img, int(row["label"])


class BrainSegDataset(Dataset):
    def __init__(self, df, transform=None):
        self.df        = df.reset_index(drop=True)
        self.transform = transform

    def __len__(self): return len(self.df)

    def __getitem__(self, idx):
        row  = self.df.iloc[idx]
        img  = cv2.imread(row["img_path"])
        img  = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        mask = cv2.imread(row["mask_path"], cv2.IMREAD_GRAYSCALE)
        mask = (mask > 127).astype(np.uint8)
        if self.transform:
            aug       = self.transform(image=img, mask=mask)
            img, mask = aug["image"], aug["mask"]
        return img, mask.long()

In [39]:
'''
class_counts   = cls_train_df['label'].value_counts().sort_index().values
class_weights  = 1. / torch.tensor(class_counts, dtype=torch.float)
sample_weights = [class_weights[label] for label in cls_train_df['label'].values]
sampler        = WeightedRandomSampler(sample_weights, len(sample_weights))

cls_train_ds = BrainClassDataset(cls_train_df, train_aug)
cls_val_ds   = BrainClassDataset(cls_val_df,   test_aug)
cls_test_ds  = BrainClassDataset(cls_test_df,  test_aug)

cls_train_dl = DataLoader(cls_train_ds, batch_size=32, sampler=sampler,  num_workers=2, pin_memory=True)
cls_val_dl   = DataLoader(cls_val_ds,   batch_size=32, shuffle=False,     num_workers=2, pin_memory=True)
cls_test_dl  = DataLoader(cls_test_ds,  batch_size=32, shuffle=False,     num_workers=2, pin_memory=True)

seg_train_ds = BrainSegDataset(seg_train_df, train_aug)
seg_test_ds  = BrainSegDataset(seg_test_df,  test_aug)

seg_train_dl = DataLoader(seg_train_ds, batch_size=16, shuffle=True,  num_workers=2, pin_memory=True)
seg_test_dl  = DataLoader(seg_test_ds,  batch_size=16, shuffle=False, num_workers=2, pin_memory=True)
'''

"\nclass_counts   = cls_train_df['label'].value_counts().sort_index().values\nclass_weights  = 1. / torch.tensor(class_counts, dtype=torch.float)\nsample_weights = [class_weights[label] for label in cls_train_df['label'].values]\nsampler        = WeightedRandomSampler(sample_weights, len(sample_weights))\n\ncls_train_ds = BrainClassDataset(cls_train_df, train_aug)\ncls_val_ds   = BrainClassDataset(cls_val_df,   test_aug)\ncls_test_ds  = BrainClassDataset(cls_test_df,  test_aug)\n\ncls_train_dl = DataLoader(cls_train_ds, batch_size=32, sampler=sampler,  num_workers=2, pin_memory=True)\ncls_val_dl   = DataLoader(cls_val_ds,   batch_size=32, shuffle=False,     num_workers=2, pin_memory=True)\ncls_test_dl  = DataLoader(cls_test_ds,  batch_size=32, shuffle=False,     num_workers=2, pin_memory=True)\n\nseg_train_ds = BrainSegDataset(seg_train_df, train_aug)\nseg_test_ds  = BrainSegDataset(seg_test_df,  test_aug)\n\nseg_train_dl = DataLoader(seg_train_ds, batch_size=16, shuffle=True,  num_wor

In [40]:
def set_resnet_grad(model, 
                    freeze_conv1=True,
                    freeze_bn1=True,
                    freeze_layer1=True,
                    freeze_layer2=True,
                    freeze_layer3=True,
                    freeze_layer4=True,
                    freeze_fc=False):
    """
    Freeze/unfreeze individual components of a torchvision ResNet.
    """
    # Stem
    for p in model.conv1.parameters():
        p.requires_grad = not freeze_conv1
    for p in model.bn1.parameters():
        p.requires_grad = not freeze_bn1

    # CE1 -> layer1
    for p in model.layer1.parameters():
        p.requires_grad = not freeze_layer1

    # CE2 -> layer2
    for p in model.layer2.parameters():
        p.requires_grad = not freeze_layer2

    # CE3 -> layer3
    for p in model.layer3.parameters():
        p.requires_grad = not freeze_layer3

    # CE4 -> layer4
    for p in model.layer4.parameters():
        p.requires_grad = not freeze_layer4

    # Classifier
    for p in model.fc.parameters():
        p.requires_grad = not freeze_fc

    # Print summary
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total     = sum(p.numel() for p in model.parameters())
    print(f"Trainable params: {trainable:,} / {total:,} ({100*trainable/total:.2f}%)")

def freeze_seg_encoder(model, model_name):
    for param in model.encoder.parameters():
        param.requires_grad = False
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total     = sum(p.numel() for p in model.parameters())
    print(f"  [{model_name}] Frozen encoder – trainable: {trainable:,} / {total:,}")

def unfreeze_seg_encoder(model):
    for param in model.parameters():
        param.requires_grad = True
    print(f"  Unfrozen – all {sum(p.numel() for p in model.parameters()):,} params trainable.")

print("Freeze/unfreeze helpers defined.")

Freeze/unfreeze helpers defined.


In [41]:
def build_efficientnet_b0():
    m = models.efficientnet_b0(weights="IMAGENET1K_V1")
    m.classifier = nn.Sequential(nn.Dropout(0.3, inplace=True), nn.Linear(m.classifier[1].in_features, NUM_CLASSES))
    return m

def build_efficientnet_b1():
    m = models.efficientnet_b1(weights="IMAGENET1K_V1")
    m.classifier = nn.Sequential(nn.Dropout(0.3, inplace=True), nn.Linear(m.classifier[1].in_features, NUM_CLASSES))
    return m

def build_efficientnet_b3():
    m = models.efficientnet_b3(weights="IMAGENET1K_V1")
    m.classifier = nn.Sequential(nn.Dropout(0.3, inplace=True), nn.Linear(m.classifier[1].in_features, NUM_CLASSES))
    return m

def build_efficientnet_v2_s():
    m = models.efficientnet_v2_s(weights="IMAGENET1K_V1")
    m.classifier = nn.Sequential(nn.Dropout(0.3, inplace=True), nn.Linear(m.classifier[1].in_features, NUM_CLASSES))
    return m

def build_resnet50():
    m = models.resnet50(weights="IMAGENET1K_V1")
    m.fc = nn.Linear(m.fc.in_features, NUM_CLASSES)
    return m

def build_mobilenet_v3():
    m = models.mobilenet_v3_small(weights="IMAGENET1K_V1")
    m.classifier[-1] = nn.Linear(m.classifier[-1].in_features, NUM_CLASSES)
    return m

def build_vgg16():
    m = models.vgg16_bn(weights="IMAGENET1K_V1")
    m.classifier[-1] = nn.Linear(m.classifier[-1].in_features, NUM_CLASSES)
    return m

def build_vgg13():
    m = models.vgg13_bn(weights="IMAGENET1K_V1")
    # Replace the classifier
    m.classifier[-1] = nn.Linear(m.classifier[-1].in_features, NUM_CLASSES)
    return m

def build_convnext_tiny():
    m = models.convnext_tiny(weights="IMAGENET1K_V1")
    m.classifier[-1] = nn.Linear(m.classifier[-1].in_features, NUM_CLASSES)
    return m

CLS_REGISTRY = {
    "EfficientNet-B0":  build_efficientnet_b0,
    "EfficientNet-B1":  build_efficientnet_b1,
    "EfficientNet-B3":  build_efficientnet_b3,
    "EfficientNetV2-S": build_efficientnet_v2_s,
    "ResNet-50":        build_resnet50,
    "MobileNet-V3-S":   build_mobilenet_v3,
    "VGG-16-BN":        build_vgg16,
    "VGG-13-BN":        build_vgg13,
    "ConvNeXt-Tiny":    build_convnext_tiny,
}

SEG_REGISTRY = {
    "UNet-EffB0":          (smp.Unet,          "efficientnet-b0"),
    "UNet-ResNet34":       (smp.Unet,          "resnet34"),
    "UNetPP-EffB0":        (smp.UnetPlusPlus,  "efficientnet-b0"),
    "UNetPP-ResNet34":     (smp.UnetPlusPlus,  "resnet34"),
    "DeepLabV3P-EffB0":    (smp.DeepLabV3Plus, "efficientnet-b0"),
    "DeepLabV3P-ResNet50": (smp.DeepLabV3Plus, "resnet50"),
    "FPN-EffB0":           (smp.FPN,           "efficientnet-b0"),
    "MAnet-EffB0":         (smp.MAnet,         "efficientnet-b0"),
}

print("CLS models:", list(CLS_REGISTRY.keys()))
print("SEG models:", list(SEG_REGISTRY.keys()))

CLS models: ['EfficientNet-B0', 'EfficientNet-B1', 'EfficientNet-B3', 'EfficientNetV2-S', 'ResNet-50', 'MobileNet-V3-S', 'VGG-16-BN', 'VGG-13-BN', 'ConvNeXt-Tiny']
SEG models: ['UNet-EffB0', 'UNet-ResNet34', 'UNetPP-EffB0', 'UNetPP-ResNet34', 'DeepLabV3P-EffB0', 'DeepLabV3P-ResNet50', 'FPN-EffB0', 'MAnet-EffB0']


In [42]:
CLS_MODEL_NAME = "ResNet-50"
SEG_MODEL_NAME = "UNetPP-ResNet34"

LR_FROZEN    = 1e-3   # higher is fine — only the head/decoder trains
LR_UNFROZEN  = 1e-4   # lower to avoid destroying pretrained weights
WEIGHT_DECAY = 1e-4

CLS_EPOCHS_FROZEN = 25
SEG_EPOCHS_FROZEN = 10

CLS_EPOCHS_UNFRZ = 25  # used in the optional Phase 2 cells
SEG_EPOCHS_UNFRZ = 10

In [43]:
cls_criterion = nn.CrossEntropyLoss()

bce_loss  = nn.BCEWithLogitsLoss()
dice_loss = smp.losses.DiceLoss(mode="binary")

def seg_criterion(pred, target):
    target_f = target.float().unsqueeze(1)
    return 0.5 * bce_loss(pred, target_f) + 0.5 * dice_loss(pred, target_f)

def dice_score(pred_logits, target):
    pred  = (torch.sigmoid(pred_logits) > 0.5).float()
    tgt   = target.float().unsqueeze(1)
    inter = (pred * tgt).sum(dim=(2, 3))
    union = pred.sum(dim=(2, 3)) + tgt.sum(dim=(2, 3))
    return ((2 * inter + 1e-6) / (union + 1e-6)).mean().item()

In [44]:
def train_one_epoch_cls(model, loader, optimizer):
    model.train()
    total_loss = correct = n = 0
    for imgs, labels in loader:
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        out  = model(imgs)
        loss = cls_criterion(out, labels)
        loss.backward(); optimizer.step()
        total_loss += loss.item() * imgs.size(0)
        correct    += (out.argmax(1) == labels).sum().item()
        n          += imgs.size(0)
    return total_loss / n, correct / n

@torch.no_grad()
def eval_cls(model, loader):
    model.eval()
    correct = n = 0
    all_preds, all_labels = [], []
    for imgs, labels in loader:
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        preds = model(imgs).argmax(1)
        correct += (preds == labels).sum().item()
        n       += imgs.size(0)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
    return correct / n, all_preds, all_labels

def train_one_epoch_seg(model, loader, optimizer):
    model.train()
    total_loss = total_dice = n = 0
    for imgs, masks in loader:
        imgs, masks = imgs.to(DEVICE), masks.to(DEVICE)
        optimizer.zero_grad()
        pred = model(imgs)
        loss = seg_criterion(pred, masks)
        loss.backward(); optimizer.step()
        total_loss += loss.item() * imgs.size(0)
        total_dice += dice_score(pred, masks) * imgs.size(0)
        n          += imgs.size(0)
    return total_loss / n, total_dice / n

@torch.no_grad()
def eval_seg(model, loader):
    model.eval()
    total_loss = total_dice = n = 0
    for imgs, masks in loader:
        imgs, masks = imgs.to(DEVICE), masks.to(DEVICE)
        pred        = model(imgs)
        total_loss += seg_criterion(pred, masks).item() * imgs.size(0)
        total_dice += dice_score(pred, masks) * imgs.size(0)
        n          += imgs.size(0)
    return total_loss / n, total_dice / n

print("Helpers ready.")

Helpers ready.


In [45]:
# ============================================================================
# CLASSIFICATION EXPERIMENT LOOP (4 Variations)
# ============================================================================

# Prepare samplers and base dataframes (these do not change)
class_counts   = cls_train_df['label'].value_counts().sort_index().values
class_weights  = 1. / torch.tensor(class_counts, dtype=torch.float)
sample_weights = [class_weights[label] for label in cls_train_df['label'].values]
sampler        = WeightedRandomSampler(sample_weights, len(sample_weights))

# Create a fixed test loader ONCE (uses test_aug = pipeline_raw)
cls_test_ds  = BrainClassDataset(cls_test_df, test_aug)
cls_test_dl  = DataLoader(cls_test_ds, batch_size=32, shuffle=False, num_workers=2, pin_memory=True)

# Storage for results comparison
cls_results_summary = {}

for pipe_name, train_pipe in PIPELINES.items():
    print("\n" + "="*80)
    print(f"RUNNING CLASSIFICATION: {pipe_name}")
    print("="*80)

    # --- 1. Create DataLoaders for this specific pipeline ---
    cls_train_ds = BrainClassDataset(cls_train_df, train_pipe)
    cls_val_ds   = BrainClassDataset(cls_val_df,   test_aug) # VALIDATION ALWAYS USES test_aug

    cls_train_dl = DataLoader(cls_train_ds, batch_size=32, sampler=sampler,  num_workers=2, pin_memory=True)
    cls_val_dl   = DataLoader(cls_val_ds,   batch_size=32, shuffle=False,     num_workers=2, pin_memory=True)

   # --- 2. Fresh Model Initialization ---
    cls_model = CLS_REGISTRY[CLS_MODEL_NAME]().to(DEVICE)

    # RESNET-50 FREEZING STRATEGY
    # 1. On gèle tout le modèle
    for param in cls_model.parameters():
        param.requires_grad = False
    
    # 2. On dégèle uniquement la couche de classification (fc pour ResNet)
    for param in cls_model.fc.parameters():
        param.requires_grad = True

    trainable = sum(p.numel() for p in cls_model.parameters() if p.requires_grad)
    total     = sum(p.numel() for p in cls_model.parameters())
    print(f"Trainable params: {trainable:,} / {total:,} ({100*trainable/total:.2f}%)")

    # --- 3. Optimizer & History ---
    cls_optimizer = torch.optim.AdamW([
        {'params': cls_model.fc.parameters(), 'lr': LR_FROZEN},
    ], weight_decay=WEIGHT_DECAY)
    
    cls_history  = {"train_loss": [], "train_acc": [], "val_acc": []}
    best_val_acc = 0.0
    best_model_path = f"best_cls_{pipe_name}.pth"

    # --- 4. Training Loop ---
    print(f"\n── Phase 1 | {CLS_MODEL_NAME} | {pipe_name} ──")
    for epoch in range(1, CLS_EPOCHS_FROZEN + 1):
        tr_loss, tr_acc = train_one_epoch_cls(cls_model, cls_train_dl, cls_optimizer)
        val_acc, _, _   = eval_cls(cls_model, cls_val_dl)

        cls_history["train_loss"].append(tr_loss)
        cls_history["train_acc"].append(tr_acc)
        cls_history["val_acc"].append(val_acc)

        flag = ""
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            torch.save(cls_model.state_dict(), best_model_path)
            flag = " ✓"

        if epoch % 5 == 0 or epoch == 1 or epoch == CLS_EPOCHS_FROZEN:
            print(f"[{pipe_name}] {epoch:02d}/{CLS_EPOCHS_FROZEN}  "
                  f"loss={tr_loss:.4f}  tr_acc={tr_acc:.4f}  val_acc={val_acc:.4f}{flag}")

    # --- 5. Final Test Evaluation ---
    cls_model.load_state_dict(torch.load(best_model_path))
    test_acc, preds, labels_ = eval_cls(cls_model, cls_test_dl)
    print(f"\n[{pipe_name}] Best Val Acc: {best_val_acc:.4f} | Test Acc: {test_acc:.4f}")

    # Store summary
    cls_results_summary[pipe_name] = {
        "best_val_acc": best_val_acc,
        "test_acc": test_acc,
        "model_path": best_model_path
    }

print("\n" + "="*80)
print("CLASSIFICATION SUMMARY")
print("="*80)
for name, res in cls_results_summary.items():
    print(f"{name:<20} | Val: {res['best_val_acc']:.4f} | Test: {res['test_acc']:.4f}")


RUNNING CLASSIFICATION: Raw
Trainable params: 8,196 / 23,516,228 (0.03%)

── Phase 1 | ResNet-50 | Raw ──
[Raw] 01/25  loss=0.6227  tr_acc=0.7925  val_acc=0.8590 ✓
[Raw] 05/25  loss=0.2808  tr_acc=0.9025  val_acc=0.8910 ✓
[Raw] 10/25  loss=0.2389  tr_acc=0.9095  val_acc=0.8990
[Raw] 15/25  loss=0.2085  tr_acc=0.9245  val_acc=0.9130 ✓
[Raw] 20/25  loss=0.1679  tr_acc=0.9417  val_acc=0.9000
[Raw] 25/25  loss=0.1778  tr_acc=0.9337  val_acc=0.9070

[Raw] Best Val Acc: 0.9130 | Test Acc: 0.8872

RUNNING CLASSIFICATION: Augmented
Trainable params: 8,196 / 23,516,228 (0.03%)

── Phase 1 | ResNet-50 | Augmented ──
[Augmented] 01/25  loss=0.7477  tr_acc=0.7290  val_acc=0.8310 ✓
[Augmented] 05/25  loss=0.4107  tr_acc=0.8420  val_acc=0.8800 ✓
[Augmented] 10/25  loss=0.3246  tr_acc=0.8832  val_acc=0.8860 ✓
[Augmented] 15/25  loss=0.3162  tr_acc=0.8842  val_acc=0.8820
[Augmented] 20/25  loss=0.3116  tr_acc=0.8800  val_acc=0.8750
[Augmented] 25/25  loss=0.2991  tr_acc=0.8952  val_acc=0.8630

[Augme

In [ ]:
# ============================================================================
# SEGMENTATION EXPERIMENT LOOP (4 Variations)
# ============================================================================

# Create a fixed test loader ONCE (uses test_aug = pipeline_raw)
seg_test_ds  = BrainSegDataset(seg_test_df, test_aug)
seg_test_dl  = DataLoader(seg_test_ds, batch_size=16, shuffle=False, num_workers=2, pin_memory=True)

seg_results_summary = {}

for pipe_name, train_pipe in PIPELINES.items():
    print("\n" + "="*80)
    print(f"RUNNING SEGMENTATION: {pipe_name}")
    print("="*80)

    # --- 1. Create DataLoaders ---
    seg_train_ds = BrainSegDataset(seg_train_df, train_pipe)
    seg_train_dl = DataLoader(seg_train_ds, batch_size=16, shuffle=True, num_workers=2, pin_memory=True)

    # --- 2. Fresh Model ---
    arch, encoder = SEG_REGISTRY[SEG_MODEL_NAME]
    seg_model     = arch(encoder_name=encoder, encoder_weights="imagenet",
                         in_channels=3, classes=1, activation=None).to(DEVICE)
    freeze_seg_encoder(seg_model, f"{SEG_MODEL_NAME}_{pipe_name}")

    seg_optimizer = torch.optim.AdamW(
        filter(lambda p: p.requires_grad, seg_model.parameters()),
        lr=LR_FROZEN, weight_decay=WEIGHT_DECAY
    )

    seg_history   = {"train_loss": [], "val_loss": [], "train_dice": [], "val_dice": []}
    best_val_dice = 0.0
    best_model_path = f"best_seg_{pipe_name}.pth"

    # --- 3. Training Loop (Short for Phase 1) ---
    print(f"\n── Phase 1 | {SEG_MODEL_NAME} | {pipe_name} ──")
    for epoch in range(1, SEG_EPOCHS_FROZEN + 1):
        tr_loss, tr_dice = train_one_epoch_seg(seg_model, seg_train_dl, seg_optimizer)
        vl_loss, vl_dice = eval_seg(seg_model, seg_test_dl)

        seg_history["train_loss"].append(tr_loss)
        seg_history["val_loss"].append(vl_loss)
        seg_history["train_dice"].append(tr_dice)
        seg_history["val_dice"].append(vl_dice)

        flag = ""
        if vl_dice > best_val_dice:
            best_val_dice = vl_dice
            torch.save(seg_model.state_dict(), best_model_path)
            flag = " ✓"

        print(f"[{pipe_name}] {epoch:02d}/{SEG_EPOCHS_FROZEN}  "
              f"tr_loss={tr_loss:.4f}  tr_dice={tr_dice:.4f}  "
              f"vl_loss={vl_loss:.4f}  vl_dice={vl_dice:.4f}{flag}")

    seg_results_summary[pipe_name] = {
        "best_val_dice": best_val_dice,
        "model_path": best_model_path
    }

print("\n" + "="*80)
print("SEGMENTATION SUMMARY")
print("="*80)
for name, res in seg_results_summary.items():
    print(f"{name:<20} | Val Dice: {res['best_val_dice']:.4f}")


RUNNING SEGMENTATION: Raw
Downloading: "https://download.pytorch.org/models/resnet34-333f7ec4.pth" to /root/.cache/torch/hub/checkpoints/resnet34-333f7ec4.pth


100%|██████████| 83.3M/83.3M [00:00<00:00, 287MB/s]


  [UNetPP-ResNet34_Raw] Frozen encoder – trainable: 4,793,937 / 26,078,609

── Phase 1 | UNetPP-ResNet34 | Raw ──
[Raw] 01/10  tr_loss=0.2155  tr_dice=0.6695  vl_loss=0.1314  vl_dice=0.7553 ✓
[Raw] 02/10  tr_loss=0.1023  tr_dice=0.7742  vl_loss=0.1287  vl_dice=0.7634 ✓
[Raw] 03/10  tr_loss=0.0892  tr_dice=0.8002  vl_loss=0.0940  vl_dice=0.7986 ✓


In [ ]:
# ============================================================================
# COMPREHENSIVE EVALUATION & VISUALIZATION FOR ALL 4 EXPERIMENTS
# ============================================================================

MEAN = np.array([0.485, 0.456, 0.406])
STD  = np.array([0.229, 0.224, 0.225])

print("\n" + "="*80)
print("FINAL EVALUATION OF ALL EXPERIMENTS")
print("="*80)

# ----------------------------------------------------------------------------
# 1. CLASSIFICATION: Compare all 4 models on Test Set
# ----------------------------------------------------------------------------
print("\n--- CLASSIFICATION TEST SET PERFORMANCE ---")
print(f"{'Pipeline':<20} | {'Test Accuracy':<15} | Best Model File")
print("-" * 60)

cls_test_results = {}

for pipe_name in PIPELINES.keys():
    model_path = f"best_cls_{pipe_name}.pth"
    
    # Fresh model load
    cls_model = CLS_REGISTRY[CLS_MODEL_NAME]().to(DEVICE)
    cls_model.load_state_dict(torch.load(model_path))
    cls_model.eval()
    
    # Evaluate
    test_acc, preds, labels_ = eval_cls(cls_model, cls_test_dl)
    cls_test_results[pipe_name] = {
        "accuracy": test_acc,
        "predictions": preds,
        "labels": labels_
    }
    print(f"{pipe_name:<20} | {test_acc:.4f}         | {model_path}")

# Find best model
best_cls_name = max(cls_test_results, key=lambda x: cls_test_results[x]["accuracy"])
print(f"\n🏆 Best Classification Model: {best_cls_name} ({cls_test_results[best_cls_name]['accuracy']:.4f})")

# ----------------------------------------------------------------------------
# 2. SEGMENTATION: Compare all 4 models on Test Set
# ----------------------------------------------------------------------------
print("\n--- SEGMENTATION TEST SET PERFORMANCE ---")
print(f"{'Pipeline':<20} | {'Test Dice':<15} | Best Model File")
print("-" * 60)

seg_test_results = {}

for pipe_name in PIPELINES.keys():
    model_path = f"best_seg_{pipe_name}.pth"
    
    # Fresh model load
    arch, encoder = SEG_REGISTRY[SEG_MODEL_NAME]
    seg_model = arch(encoder_name=encoder, encoder_weights=None, # Weights loaded from state_dict
                     in_channels=3, classes=1, activation=None).to(DEVICE)
    seg_model.load_state_dict(torch.load(model_path))
    seg_model.eval()
    
    # Evaluate
    _, test_dice = eval_seg(seg_model, seg_test_dl)
    seg_test_results[pipe_name] = {
        "dice": test_dice,
        "model": seg_model
    }
    print(f"{pipe_name:<20} | {test_dice:.4f}         | {model_path}")

# Find best model
best_seg_name = max(seg_test_results, key=lambda x: seg_test_results[x]["dice"])
print(f"\n🏆 Best Segmentation Model: {best_seg_name} ({seg_test_results[best_seg_name]['dice']:.4f})")

# ----------------------------------------------------------------------------
# 3. VISUALIZATION: Classification Confusion Matrices (2x2 Grid)
# ----------------------------------------------------------------------------
fig, axes = plt.subplots(2, 2, figsize=(14, 12))
axes = axes.flatten()

for idx, (pipe_name, results) in enumerate(cls_test_results.items()):
    cm = confusion_matrix(results["labels"], results["predictions"])
    disp = ConfusionMatrixDisplay(cm, display_labels=CLASS_NAMES)
    disp.plot(ax=axes[idx], colorbar=False, xticks_rotation=30, cmap='Blues')
    axes[idx].set_title(f"{pipe_name} (Acc: {results['accuracy']:.4f})")
    
plt.suptitle(f"{CLS_MODEL_NAME} - Confusion Matrices Comparison", fontsize=16, y=1.02)
plt.tight_layout()
plt.savefig("cls_cm_comparison_all.png", dpi=150, bbox_inches="tight")
plt.show()

# ----------------------------------------------------------------------------
# 4. VISUALIZATION: Classification Accuracy Curves Comparison
# ----------------------------------------------------------------------------
# Note: We need to re-run training briefly to capture history, 
# or we load from saved histories. Since we didn't save history dicts, 
# we'll create a bar chart comparison instead.

fig, ax = plt.subplots(figsize=(10, 6))
names = list(cls_test_results.keys())
accs = [cls_test_results[n]["accuracy"] for n in names]
colors = ['#2ecc71', '#3498db', '#9b59b6', '#e74c3c']

bars = ax.bar(names, accs, color=colors, edgecolor='black', linewidth=1.5)
ax.set_ylabel("Test Accuracy")
ax.set_title(f"{CLS_MODEL_NAME} - Test Accuracy by Preprocessing Pipeline")
ax.set_ylim([0.85, 1.0])
ax.grid(axis='y', alpha=0.3)

# Add value labels on bars
for bar, acc in zip(bars, accs):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005, 
            f'{acc:.4f}', ha='center', va='bottom', fontweight='bold')

plt.tight_layout()
plt.savefig("cls_accuracy_comparison.png", dpi=150)
plt.show()

# ----------------------------------------------------------------------------
# 5. VISUALIZATION: Segmentation Dice Scores Comparison
# ----------------------------------------------------------------------------
fig, ax = plt.subplots(figsize=(10, 6))
names = list(seg_test_results.keys())
dices = [seg_test_results[n]["dice"] for n in names]

bars = ax.bar(names, dices, color=colors, edgecolor='black', linewidth=1.5)
ax.set_ylabel("Test Dice Score")
ax.set_title(f"{SEG_MODEL_NAME} - Test Dice by Preprocessing Pipeline")
ax.set_ylim([0.70, 0.90])
ax.grid(axis='y', alpha=0.3)

for bar, dice in zip(bars, dices):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005, 
            f'{dice:.4f}', ha='center', va='bottom', fontweight='bold')

plt.tight_layout()
plt.savefig("seg_dice_comparison.png", dpi=150)
plt.show()

# ----------------------------------------------------------------------------
# 6. VISUALIZATION: Qualitative Segmentation Comparison (Best Model Only)
# ----------------------------------------------------------------------------
best_seg_model = seg_test_results[best_seg_name]["model"]
sample_ds = BrainSegDataset(seg_test_df.sample(4, random_state=42).reset_index(drop=True), test_aug)

fig, axes = plt.subplots(4, 3, figsize=(12, 16))
best_seg_model.eval()

with torch.no_grad():
    for i in range(4):
        img_t, mask_t = sample_ds[i]
        pred = torch.sigmoid(best_seg_model(img_t.unsqueeze(0).to(DEVICE))).squeeze().cpu().numpy()
        img_np = (img_t.permute(1, 2, 0).numpy() * STD + MEAN).clip(0, 1)
        dice = (2 * ((pred > 0.5) * mask_t.numpy()).sum() + 1e-6) / \
               ((pred > 0.5).sum() + mask_t.numpy().sum() + 1e-6)
        
        axes[i, 0].imshow(img_np)
        axes[i, 0].set_title("MRI Input")
        axes[i, 1].imshow(mask_t.numpy(), cmap="hot")
        axes[i, 1].set_title("Ground Truth")
        axes[i, 2].imshow(pred > 0.5, cmap="hot")
        axes[i, 2].set_title(f"Pred (Dice={dice:.3f})")
        
        for ax in axes[i]:
            ax.axis("off")

plt.suptitle(f"Best Segmentation Model: {best_seg_name} (Test Dice: {seg_test_results[best_seg_name]['dice']:.4f})", 
             fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig(f"seg_qual_best_{best_seg_name}.png", dpi=150, bbox_inches="tight")
plt.show()

# ----------------------------------------------------------------------------
# 7. DETAILED REPORT: Best Classification Model
# ----------------------------------------------------------------------------
print("\n" + "="*80)
print(f"DETAILED CLASSIFICATION REPORT - BEST MODEL: {best_cls_name}")
print("="*80)
best_cls_preds = cls_test_results[best_cls_name]["predictions"]
best_cls_labels = cls_test_results[best_cls_name]["labels"]
print(classification_report(best_cls_labels, best_cls_preds, target_names=CLASS_NAMES))

print("\n✅ Full comparison complete!")
print(f"   Best Classification: {best_cls_name} ({cls_test_results[best_cls_name]['accuracy']:.4f})")
print(f"   Best Segmentation:  {best_seg_name} ({seg_test_results[best_seg_name]['dice']:.4f})")